In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [2]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']

    prompt = f'Generate a detailed outline for a blog on the topic: {title}'

    outline = model.invoke(prompt).content

    state['outline'] = outline
    return state

In [3]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content
    return state

In [4]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

initial_state = {'title': 'Rise of AI in India'}
final_state = workflow.invoke(initial_state)

print(final_state)
print(final_state['outline'])   # the generated outline
print(final_state['content'])   # the full blog

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'title': 'Rise of AI in India', 'outline': [{'type': 'text', 'text': 'Here is a comprehensive and detailed blog outline on the **"Rise of AI in India."** It is structured to be engaging, SEO-friendly, and easy to expand into a full blog post (1,500–2,500 words).\n\n---\n\n# Blog Title Ideas:\n1. **From IT Hub to AI Powerhouse: The Rise of Artificial Intelligence in India**\n2. **India’s AI Revolution: How Technology is Reshaping the World’s Most Populous Nation**\n3. **#AIforAll: Understanding the Growth, Impact, and Future of AI in India**\n\n---\n\n## I. Introduction\n* **The Hook:** Start with a compelling stat (e.g., *NASSCOM estimates AI could add $450–$500 billion to India’s GDP by 2025* or the surge in Indian AI startups).\n* **Context:** Briefly explain India\'s transition from being the "back office of the world" (IT outsourcing) to a front-runner in deep-tech and AI innovation.\n* **Thesis Statement:** India\'s unique combination of massive data availability, strong digital 